# Laboratory 00 — How many makes a law

In this laboratory you will watch a reproducible number assemble itself out of completely
unpredictable ones, measure exactly how fast it becomes reproducible, and falsify the
explanation most people reach for first.

There is no physics here on purpose. Dice have no energy, no container and no dynamics, so
whatever makes their average settle down cannot be a physical mechanism — it has to be the
arithmetic of many. That arithmetic is what modules 3, 4 and 8 will reuse.

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell. That is not a ritual: a prediction you
have committed to is the only reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | $N$ dice, each showing one of six equally likely faces; the observable is their sum or their average |
| **Dynamics** | none — each draw is independent of every other, with no memory between them |
| **Boundary** | closed trivially; $N$ is fixed for one experiment and nothing is exchanged |
| **Ensemble** | every face equally probable, by assumption rather than by proof |
| **Ignored** | all physics of real dice — the tumble, the bounce, pip asymmetry, air drag |
| **Valid when** | the randomness is well described by independent uniform draws |
| **Failure modes** | correlated draws, loaded dice, any question about a *single* roll |

All the model code lives in `thermolab.sampling` and `thermolab.forms` — open them and read
them. Nothing in this course is hidden inside a framework.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import forms, sampling
from thermolab.validation import relative_error, scaling_exponent, seed_study

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"one fair die:  mean = {sampling.die_mean():.4f}")
print(f"               variance = {sampling.die_variance():.4f}")
print(f"               sigma = {np.sqrt(sampling.die_variance()):.4f}")
print(f"               sigma/mean = {sampling.die_relative_spread():.4f}  <- the coefficient")

## Part 1 — Watch a law assemble itself

Six hundred rolls of one die, in order, beside the running average of that very same
sequence.

In [ ]:
rolls = sampling.roll_dice(600, rng)
curve = sampling.running_average(rolls)
trials = np.arange(1, rolls.size + 1)

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.8))

left.scatter(trials, rolls, s=14, alpha=0.6)
left.axhline(sampling.die_mean(), color="crimson", ls="--")
left.set_yticks(range(1, 7))
left.set_xlabel("roll number")
left.set_ylabel("face shown")
left.set_title("the microscopic reality")

right.plot(trials, curve, lw=1.4)
right.axhline(sampling.die_mean(), color="crimson", ls="--")
right.set_ylim(1, 6)
right.set_xlabel("rolls averaged, N")
right.set_ylabel("average so far")
right.set_title("the same data, accumulated")

plt.tight_layout()
plt.show()

print(f"average of all {rolls.size} rolls: {rolls.mean():.4f}   (exact answer 3.5)")

The left panel never becomes orderly. It cannot: each roll is drawn from exactly the same
distribution as the first one, and nothing about the sequence changes as it goes on.

### Predict

The next cell repeats the experiment "average $N$ dice" many times over, at $N = 10$, $100$
and $1000$, and plots one dot per completed experiment. Before running it, commit to an
answer:

- Roughly how far from $3.5$ will a typical dot sit at $N = 10$? At $N = 1000$?
- By what factor should the vertical spread shrink between those two panels?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
sizes = (10, 100, 1000)
n_experiments = 240
columns = [sampling.sample_averages(n, n_experiments, rng) for n in sizes]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, n_per_sample, averages in zip(axes, sizes, columns, strict=True):
    ax.scatter(np.arange(1, n_experiments + 1), averages, s=10, alpha=0.6)
    ax.axhline(sampling.die_mean(), color="crimson", ls="--")
    ax.set_ylim(2.3, 4.7)
    ax.set_title(f"N = {n_per_sample}")
    ax.set_xlabel("experiment number")
axes[0].set_ylabel("average of N dice")
plt.tight_layout()
plt.show()

for n_per_sample, averages in zip(sizes, columns, strict=True):
    print(f"N = {n_per_sample:5d}   measured spread = {averages.std(ddof=1):.4f}"
          f"   predicted = {np.sqrt(sampling.die_variance() / n_per_sample):.4f}")

Each panel shares a vertical scale, so the narrowing is real rather than an artefact of
axis limits. A hundredfold increase in $N$ narrowed the cloud tenfold — which is the
$N^{-1/2}$ law, seen before it has been derived.

## Part 2 — Measure the exponent, and the coefficient

A fitted exponent on its own is weak evidence: plenty of wrong models produce a slope near
$-\tfrac{1}{2}$. So we check the prefactor too. The derivation predicts

$$
\frac{\sigma_{\bar{A}_N}}{\mu_1} = \frac{\sigma_1}{\mu_1} \, N^{-1/2} ,
\qquad \frac{\sigma_1}{\mu_1} \approx 0.488 .
$$

In [ ]:
fit_sizes = np.array([4, 16, 64, 256, 1024])
spreads = np.array([
    sampling.relative_spread_of_average(int(n), n_samples=1500, rng=rng) for n in fit_sizes
])
predicted = np.array([sampling.predicted_relative_spread(int(n)) for n in fit_sizes])

exponent = scaling_exponent(fit_sizes, spreads)
coefficients = spreads * np.sqrt(fit_sizes)

print(f"fitted exponent: {exponent:+.4f}   (predicted -0.5)\n")
print(f"{'N':>6}  {'measured':>10}  {'predicted':>10}  {'rel. error':>10}  {'coeff':>7}")
for n, measured, expected, coefficient in zip(fit_sizes, spreads, predicted, coefficients,
                                              strict=True):
    print(f"{n:6d}  {measured:10.5f}  {expected:10.5f}"
          f"  {relative_error(measured, expected):10.3f}  {coefficient:7.4f}")
print(f"\ncoefficient should approach sigma_1/mu_1 = {sampling.die_relative_spread():.4f}")

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.loglog(fit_sizes, spreads, "o", label="measured")
plt.loglog(fit_sizes, predicted, "-", label=r"$(\sigma_1/\mu_1)\,N^{-1/2}$")
plt.xlabel("N")
plt.ylabel(r"$\sigma_{\bar{A}} / \mu$")
plt.title(f"relative spread of the average (fitted slope {exponent:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

Note that the fitted line matches in *height* as well as slope. That is the stronger claim,
and it is the one that would break if the derivation were wrong in a way a slope check
could not see.

### The sum goes the other way

The same data, read differently. Watch the absolute scatter of the sum grow while the
relative scatter of the average shrinks.

In [ ]:
sum_spreads = []
average_spreads = []
for n in fit_sizes:
    averages = sampling.sample_averages(int(n), 1500, rng)
    sum_spreads.append(float((n * averages).std(ddof=1)))
    average_spreads.append(float(averages.std(ddof=1) / averages.mean()))

print(f"sum, absolute scatter      fitted exponent {scaling_exponent(fit_sizes, sum_spreads):+.3f}"
      "   (predicted +0.5)")
print(f"average, relative scatter  fitted exponent "
      f"{scaling_exponent(fit_sizes, average_spreads):+.3f}   (predicted -0.5)")

## Part 3 — Dilution, not compensation

Here is the experiment worth running yourself. The usual explanation for why an average
settles is that early deviations get cancelled by later ones. If that were true, then runs
that *started* unusually high would have to continue unusually low.

We can just check. Draw many runs, keep only those whose first ten rolls averaged above
$4.5$, and then look at what those same runs did next.

In [ ]:
n_runs, n_start, n_after = 40_000, 10, 60

starts = rng.integers(1, 7, size=(n_runs, n_start))
afters = rng.integers(1, 7, size=(n_runs, n_after))

hot = starts.mean(axis=1) > 4.5
print(f"{hot.sum()} of {n_runs} runs started hot (first {n_start} rolls averaged > 4.5)")
print(f"  their opening average:                  {starts[hot].mean():.4f}")

after_mean = afters[hot].mean()
after_error = afters[hot].std(ddof=1) / np.sqrt(afters[hot].size)
print(f"  the SAME runs, over the next {n_after} rolls: {after_mean:.4f} +/- {after_error:.4f}")
print(f"  every run, over the next {n_after} rolls:     {afters.mean():.4f}")
print("\nIf the dice compensated, the middle number would sit below 3.5. It does not.")

In [ ]:
# Where the early excess actually goes: it is divided away, never cancelled.
combined = np.hstack([starts[hot], afters[hot]])
mean_curve = np.array([sampling.running_average(run) for run in combined]).mean(axis=0)
k = np.arange(1, combined.shape[1] + 1)

# The comparison curve assumes every roll after the opening block averages exactly 3.5 — no
# correction of any kind — and lets the opening excess be divided away. It is only defined
# once that block is complete, so it starts at k = n_start.
tail = k >= n_start
dilution = (n_start * starts[hot].mean() + (k[tail] - n_start) * sampling.die_mean()) / k[tail]

plt.figure(figsize=(7, 4.2))
plt.plot(k, mean_curve, lw=1.6, label="hot-start runs, mean running average")
plt.plot(k[tail], dilution, "--", lw=1.2, label="pure dilution: excess never corrected")
plt.axhline(3.5, color="crimson", ls=":", label="3.5")
plt.xlabel("rolls averaged, N")
plt.ylabel("average so far")
plt.title("an early excess is divided away, not cancelled")
plt.legend()
plt.tight_layout()
plt.show()

The two curves lie on top of each other. The dashed line was computed by assuming every
roll after the tenth averages exactly $3.5$ — no correction whatsoever — and it reproduces
the measurement. The dice have no memory; the early excess is a fixed quantity being
divided by an ever-larger $N$.

This is the misconception the module's first quiz question is built on, and it is worth
noticing that it predicts the *right number* for the wrong reason. Getting the exponent
right does not mean you have the mechanism right.

## Part 4 — Exact and inexact, as pure mathematics

The other half of this module's diagnostic is calculus. This part has nothing to do with
dice; it is the piece of mathematics module 5 will be built on, met here in its simplest
form.

A differential form $\omega = M\,\mathrm{d}x + N\,\mathrm{d}y$ becomes a number only once
you give it a path. Sometimes the number depends on the path and sometimes it does not.

In [ ]:
def route_diagonal(n_points):
    t = np.linspace(0.0, 1.0, n_points)
    return t, t


def route_across_then_up(n_points):
    t = np.linspace(0.0, 1.0, n_points)
    x = np.hstack([t, np.ones(n_points)])
    y = np.hstack([np.zeros(n_points), t])
    return x, y


def route_up_then_across(n_points):
    t = np.linspace(0.0, 1.0, n_points)
    x = np.hstack([np.zeros(n_points), t])
    y = np.hstack([t, np.ones(n_points)])
    return x, y


ROUTES = {
    "diagonal y = x": route_diagonal,
    "across, then up": route_across_then_up,
    "up, then across": route_up_then_across,
}

# omega_1 = y dx + x dy  is d(xy);   omega_2 = y dx  is the differential of nothing.
FORMS = {
    "w1 = y dx + x dy": (lambda x, y: y, lambda x, y: x),
    "w2 = y dx": (lambda x, y: y, lambda x, y: np.zeros_like(x)),
}

print(f"{'':20}" + "".join(f"{name:>18}" for name in ROUTES))
for form_name, (m, n) in FORMS.items():
    values = [forms.line_integral(m, n, *route(801)) for route in ROUTES.values()]
    print(f"{form_name:20}" + "".join(f"{v:18.4f}" for v in values))

In [ ]:
# The criterion decides it without integrating anything at all.
probe_x = np.linspace(0.2, 1.8, 40)
probe_y = np.linspace(0.3, 1.7, 40)

for form_name, (m, n) in FORMS.items():
    gap = forms.mixed_partials_gap(m, n, probe_x, probe_y)
    verdict = "exact" if forms.is_exact(m, n, probe_x, probe_y) else "INEXACT"
    print(f"{form_name:20}  dM/dy - dN/dx = {gap.mean():+.6f}   -> {verdict}")

The first form gives $1$ on every route, because it is $\mathrm{d}(xy)$ and the answer is
just $f(1,1) - f(0,0)$. The second gives $0$, $\tfrac{1}{2}$ or $1$ depending on how you
travelled — same start, same finish, three answers.

In module 5 the plane becomes the $P$–$V$ plane. The exact form integrates to a change in
internal energy, which depends only on the endpoints; the inexact one integrates to work,
which depends on the route. Nothing physical is needed to see the difference — it is
already here, in $y\,\mathrm{d}x$.

## Part 5 — Automated checks

Every claim above is also a test in the project's test suite. These are the same checks,
run here so that a notebook which silently stops being true fails loudly.

In [ ]:
# 1. The die's exact moments — no approximation, no sampling.
assert sampling.die_mean(6) == 3.5
assert abs(sampling.die_variance(6) - 35 / 12) < 1e-12

# 2. Conservation of counting: every roll lands on a real face.
sample = sampling.roll_dice(5000, np.random.default_rng(1))
assert sample.min() >= 1 and sample.max() <= 6
assert np.bincount(sample, minlength=7).sum() == sample.size

# 3. The running average ends exactly on the plain mean — no drift introduced.
assert abs(sampling.running_average(sample)[-1] - sample.mean()) < 1e-12

# 4. The N^(-1/2) law, with an honest error bar across independent seeds.
study = seed_study(
    lambda r: sampling.relative_spread_of_average(64, 1500, r), n_seeds=8
)
target = sampling.predicted_relative_spread(64)
assert study.agrees_with(target, n_sigma=3.0)

# 5. Path-independence of an exact form, and its failure for an inexact one.
diagonal = forms.line_integral(lambda x, y: y, lambda x, y: x, *route_diagonal(801))
stepped = forms.line_integral(lambda x, y: y, lambda x, y: x, *route_across_then_up(801))
assert abs(diagonal - stepped) < 1e-9
inexact_a = forms.line_integral(lambda x, y: y, lambda x, y: np.zeros_like(x),
                                *route_diagonal(801))
inexact_b = forms.line_integral(lambda x, y: y, lambda x, y: np.zeros_like(x),
                                *route_across_then_up(801))
assert abs(inexact_a - inexact_b) > 0.4

print(f"exact moments           mean {sampling.die_mean():.4f}, "
      f"variance {sampling.die_variance():.6f}")
print(f"relative spread at N=64 {study.mean:.5f} +/- {study.standard_error:.5f}"
      f"  vs predicted {target:.5f}")
print(f"exact form, two routes  {diagonal:.6f} and {stepped:.6f}  (must agree)")
print(f"inexact form, two routes {inexact_a:.6f} and {inexact_b:.6f}  (must differ)")
print("\nall checks passed")

## Part 6 — Explore it yourself

Set the sliders, then press **Run** to redraw. Two things worth doing:

1. Set the die to two faces (a coin, scored 1 and 2). The coefficient $\sigma_1/\mu_1$
   changes; check that the *exponent* does not.
2. Push `n_per_sample` up and watch the dot cloud collapse onto the dashed line. There is
   no value at which the behaviour changes character — it just keeps narrowing.

In [ ]:
import ipywidgets as widgets


# interact_manual, not interact: set all three sliders, then press Run. Besides being the
# saner interaction for a callback that redraws two panels, it keeps this notebook fast under
# automated execution — a figure emitted from inside a live `interact` callback costs about
# five minutes per notebook under nbmake on Windows, against four seconds this way.
def explore(n_per_sample=100, n_experiments=200, n_faces=6):
    local = np.random.default_rng(0)
    averages = sampling.sample_averages(n_per_sample, n_experiments, local, n_faces)
    mean = sampling.die_mean(n_faces)
    measured = averages.std(ddof=1) / averages.mean()
    expected = sampling.predicted_relative_spread(n_per_sample, n_faces)

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
    left.scatter(np.arange(1, n_experiments + 1), averages, s=10, alpha=0.6)
    left.axhline(mean, color="crimson", ls="--")
    left.set_xlabel("experiment number")
    left.set_ylabel(f"average of {n_per_sample} draws")
    left.set_title(f"measured spread {measured:.4f} vs predicted {expected:.4f}")

    right.plot(np.arange(1, n_per_sample + 1),
               sampling.running_average(sampling.roll_dice(n_per_sample, local, n_faces)), lw=1.2)
    right.axhline(mean, color="crimson", ls="--")
    right.set_xlabel("rolls averaged, N")
    right.set_ylabel("average so far")
    right.set_title("one run, accumulating")
    plt.tight_layout()
    plt.show()


widgets.interact_manual(
    explore,
    n_per_sample=widgets.IntSlider(min=2, max=2000, step=2, value=100, description="N"),
    n_experiments=widgets.IntSlider(min=50, max=600, step=25, value=200, description="repeats"),
    n_faces=widgets.IntSlider(min=2, max=20, step=1, value=6, description="faces"),
);

## Check your understanding

This quiz is a diagnostic, not an exam. Every item's feedback names where to go if it
caught you.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "00-orientation.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. Part 3 falsified one explanation for why averages settle. State the explanation that
   survived, in one sentence, without using the word "cancel".
3. The generator used here produces independent draws *by construction*. Name one
   conclusion from today that is therefore **not** established by these numbers, even
   though everything agreed.
4. Which of the four skills — units, conventions, derivatives, probability — cost you the
   most today? What will you do about it before module 1?

**Your answers:**

1.
2.
3.
4.